In [1]:
#Load libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import mean_absolute_error, r2_score,mean_squared_error
from sklearn.linear_model import LinearRegression

In [2]:
#Predict round of 32
file1 = pd.read_excel("C:/Users/yashs/output5.xlsx")

In [3]:
file1.columns

Index(['match_id', 'home_team', 'home_ranking', 'away_team', 'away_ranking',
       'home_score', 'home_xg', 'home_penalty', 'away_score', 'away_xg',
       'corners_away', 'corners_home', 'fouls_away', 'fouls_home',
       'shots_on_target_away', 'shots_on_target_home', 'possession_away',
       'possession_home', 'away_penalty', 'Date', 'Year', 'home_own_goal.1',
       'away_own_goal.1', 'home_red_card.1', 'away_red_card.1',
       'home_yellow_red_card.1', 'away_yellow_red_card.1',
       'h2h_vs_top10winrate_brazil.1', 'knockout_stage_reach_rate_brazil.1',
       'h2h_vs_top10winrate_argentina.1',
       'knockout_stage_reach_rate_argentina.1',
       'h2h_vs_top10winrate_germany.1', 'knockout_stage_reach_rate_germany.1',
       'h2h_vs_top10winrate_france.1', 'knockout_stage_reach_rate_france.1',
       'h2h_vs_top10winrate_spain.1', 'knockout_stage_reach_rate_spain.1',
       'h2h_vs_top10winrate_england.1', 'knockout_stage_reach_rate_england.1',
       'h2h_vs_top10winrate_mexi

In [4]:
df = file1.drop(columns=['DaysSinceLastHomeMatch', 'DaysSinceLastAwayMatch','result', 'home_team_avg_goals_past', 'away_team_avg_goals_past',  'home_team_avg_goals_last5', 'away_team_avg_goals_last5'])

In [5]:
df.columns

Index(['match_id', 'home_team', 'home_ranking', 'away_team', 'away_ranking',
       'home_score', 'home_xg', 'home_penalty', 'away_score', 'away_xg',
       'corners_away', 'corners_home', 'fouls_away', 'fouls_home',
       'shots_on_target_away', 'shots_on_target_home', 'possession_away',
       'possession_home', 'away_penalty', 'Date', 'Year', 'home_own_goal.1',
       'away_own_goal.1', 'home_red_card.1', 'away_red_card.1',
       'home_yellow_red_card.1', 'away_yellow_red_card.1',
       'h2h_vs_top10winrate_brazil.1', 'knockout_stage_reach_rate_brazil.1',
       'h2h_vs_top10winrate_argentina.1',
       'knockout_stage_reach_rate_argentina.1',
       'h2h_vs_top10winrate_germany.1', 'knockout_stage_reach_rate_germany.1',
       'h2h_vs_top10winrate_france.1', 'knockout_stage_reach_rate_france.1',
       'h2h_vs_top10winrate_spain.1', 'knockout_stage_reach_rate_spain.1',
       'h2h_vs_top10winrate_england.1', 'knockout_stage_reach_rate_england.1',
       'h2h_vs_top10winrate_mexi

In [6]:
df = df.drop(columns =['match_id', 'goal_difference','rank_difference'])

In [7]:
df.columns

Index(['home_team', 'home_ranking', 'away_team', 'away_ranking', 'home_score',
       'home_xg', 'home_penalty', 'away_score', 'away_xg', 'corners_away',
       'corners_home', 'fouls_away', 'fouls_home', 'shots_on_target_away',
       'shots_on_target_home', 'possession_away', 'possession_home',
       'away_penalty', 'Date', 'Year', 'home_own_goal.1', 'away_own_goal.1',
       'home_red_card.1', 'away_red_card.1', 'home_yellow_red_card.1',
       'away_yellow_red_card.1', 'h2h_vs_top10winrate_brazil.1',
       'knockout_stage_reach_rate_brazil.1', 'h2h_vs_top10winrate_argentina.1',
       'knockout_stage_reach_rate_argentina.1',
       'h2h_vs_top10winrate_germany.1', 'knockout_stage_reach_rate_germany.1',
       'h2h_vs_top10winrate_france.1', 'knockout_stage_reach_rate_france.1',
       'h2h_vs_top10winrate_spain.1', 'knockout_stage_reach_rate_spain.1',
       'h2h_vs_top10winrate_england.1', 'knockout_stage_reach_rate_england.1',
       'h2h_vs_top10winrate_mexico.1', 'knockout_st

In [8]:
df['Date'] = pd.to_datetime(df['Date'])

In [9]:
df= df.sort_values(by=['Date'])

In [10]:
df['DaysSinceLastHomeMatch'] = df.groupby('home_team')['Date'].diff().dt.days

In [11]:
df['DaysSinceLastAwayMatch'] = df.groupby('away_team')['Date'].diff().dt.days

In [12]:
df['rank_difference'] = (df['home_ranking'] - df['away_ranking'])

In [13]:
df["goal_difference"] = df["home_score"] - df["away_score"]

In [14]:
df['result'] = np.select(
    [df['home_score'] > df['away_score'],
     df['home_score'] < df['away_score']],
    ['Win', 'Loss'],
    default='Draw'
)

In [15]:
df = df.sort_values("Date").reset_index(drop=True)
df = df.reset_index().rename(columns={"index": "match_id"})

# Long format: one row per team appearance, in date order
home = df[["match_id", "Date", "home_team", "home_score"]].rename(
    columns={"home_team": "team", "home_score": "goals"})
home["side"] = "home"
away = df[["match_id", "Date", "away_team", "away_score"]].rename(
    columns={"away_team": "team", "away_score": "goals"})
away["side"] = "away"
long = pd.concat([home, away]).sort_values(["Date", "match_id"])

# Expanding average of past matches only — shift(1) excludes the current match
long["avg_goals_past"] = long.groupby("team")["goals"].transform(
    lambda s: s.shift(1).expanding().mean()
)

# Rolling average of the last 5 matches (recent form)
long["avg_goals_last5"] = long.groupby("team")["goals"].transform(
    lambda s: s.shift(1).rolling(5, min_periods=1).mean()
)

# Map back onto the original dataframe
wide = long.pivot(index="match_id", columns="side", values=["avg_goals_past", "avg_goals_last5"])
df["home_team_avg_goals_past"] = wide[("avg_goals_past", "home")]
df["away_team_avg_goals_past"] = wide[("avg_goals_past", "away")]
df["home_team_avg_goals_last5"] = wide[("avg_goals_last5", "home")]
df["away_team_avg_goals_last5"] = wide[("avg_goals_last5", "away")]

In [16]:
df.to_excel('output25.xlsx', index = False)

<ipython-input-16-f2747a95c0c3>:1: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.8' currently installed).
  df.to_excel('output25.xlsx', index = False)


In [17]:
file2 = pd.read_excel("C:/Users/yashs/output25.xlsx")

In [18]:
file2['Date'] = pd.to_datetime(file2['Date'])

In [19]:
cols = ["home_team_avg_goals_past", "away_team_avg_goals_past",
        "home_team_avg_goals_last5", "away_team_avg_goals_last5"]
file2[cols] = file2[cols].fillna(file2[cols].mean())

In [20]:
df1= file2.sort_values(by=['Date'])

In [21]:
test = df1[df1['Year'] < 2025]

In [22]:
test.columns

Index(['match_id', 'home_team', 'home_ranking', 'away_team', 'away_ranking',
       'home_score', 'home_xg', 'home_penalty', 'away_score', 'away_xg',
       'corners_away', 'corners_home', 'fouls_away', 'fouls_home',
       'shots_on_target_away', 'shots_on_target_home', 'possession_away',
       'possession_home', 'away_penalty', 'Date', 'Year', 'home_own_goal.1',
       'away_own_goal.1', 'home_red_card.1', 'away_red_card.1',
       'home_yellow_red_card.1', 'away_yellow_red_card.1',
       'h2h_vs_top10winrate_brazil.1', 'knockout_stage_reach_rate_brazil.1',
       'h2h_vs_top10winrate_argentina.1',
       'knockout_stage_reach_rate_argentina.1',
       'h2h_vs_top10winrate_germany.1', 'knockout_stage_reach_rate_germany.1',
       'h2h_vs_top10winrate_france.1', 'knockout_stage_reach_rate_france.1',
       'h2h_vs_top10winrate_spain.1', 'knockout_stage_reach_rate_spain.1',
       'h2h_vs_top10winrate_england.1', 'knockout_stage_reach_rate_england.1',
       'h2h_vs_top10winrate_mexi

In [23]:
test= test.drop(columns = ['match_id'])

In [24]:
target = test['result']
target

0       Win
1       Win
2       Win
3       Win
4       Win
       ... 
458    Loss
460     Win
461     Win
462     Win
463    Draw
Name: result, Length: 464, dtype: object

In [25]:
test = test.drop(columns =['result'])

In [26]:
test = test.drop(columns=['home_team', 'away_team','home_score','away_score'])

In [27]:
test = test.replace('', 0).fillna(0)

In [28]:
test = test.drop(columns = ['Date'])

In [29]:
test.isnull().sum()

home_ranking                 0
away_ranking                 0
home_xg                      0
home_penalty                 0
away_xg                      0
                            ..
goal_difference              0
home_team_avg_goals_past     0
away_team_avg_goals_past     0
home_team_avg_goals_last5    0
away_team_avg_goals_last5    0
Length: 90, dtype: int64

In [ ]:
test.columns

In [ ]:
test.drop(columns = ['home_xg', 'home_penalty', 'away_xg',
       'corners_away', 'corners_home', 'fouls_away', 'fouls_home',
       'shots_on_target_away', 'shots_on_target_home', 'possession_away',
       'possession_home', 'away_penalty'])

In [ ]:
test.drop(columns = ['home_own_goal.1',
       'away_own_goal.1', 'home_red_card.1', 'away_red_card.1',
       'home_yellow_red_card.1', 'away_yellow_red_card.1'])

In [30]:
scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(test), 
    columns=test.columns, 
    index=test.index
)

In [31]:
X_train, X_test, y_train, y_test = train_test_split(df_scaled, target, test_size=0.3, stratify = target)

In [32]:
df_scaled.columns

Index(['home_ranking', 'away_ranking', 'home_xg', 'home_penalty', 'away_xg',
       'corners_away', 'corners_home', 'fouls_away', 'fouls_home',
       'shots_on_target_away', 'shots_on_target_home', 'possession_away',
       'possession_home', 'away_penalty', 'Year', 'home_own_goal.1',
       'away_own_goal.1', 'home_red_card.1', 'away_red_card.1',
       'home_yellow_red_card.1', 'away_yellow_red_card.1',
       'h2h_vs_top10winrate_brazil.1', 'knockout_stage_reach_rate_brazil.1',
       'h2h_vs_top10winrate_argentina.1',
       'knockout_stage_reach_rate_argentina.1',
       'h2h_vs_top10winrate_germany.1', 'knockout_stage_reach_rate_germany.1',
       'h2h_vs_top10winrate_france.1', 'knockout_stage_reach_rate_france.1',
       'h2h_vs_top10winrate_spain.1', 'knockout_stage_reach_rate_spain.1',
       'h2h_vs_top10winrate_england.1', 'knockout_stage_reach_rate_england.1',
       'h2h_vs_top10winrate_mexico.1', 'knockout_stage_reach_rate_mexico.1',
       'h2h_vs_top10winrate_netherla

In [76]:
rf = RandomForestClassifier(n_estimators = 1,  min_samples_split = 3, class_weight = 'balanced')
rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', min_samples_split=3,
                       n_estimators=1)

In [79]:
rf1 = RandomForestClassifier(n_estimators = 2,  min_samples_split = 2, class_weight = 'balanced')
rf1.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=2)

In [80]:
y_pred = rf1.predict(X_test)

In [81]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.7785714285714286
Confusion Matrix:
 [[26  5  0]
 [ 6 30  0]
 [16  4 53]]

Classification Report:
               precision    recall  f1-score   support

        Draw       0.54      0.84      0.66        31
        Loss       0.77      0.83      0.80        36
         Win       1.00      0.73      0.84        73

    accuracy                           0.78       140
   macro avg       0.77      0.80      0.77       140
weighted avg       0.84      0.78      0.79       140



In [82]:
#Predictions
file2 = pd.read_excel("C:/Users/yashs/output25.xlsx")

In [83]:
file2['Date'] = pd.to_datetime(file2['Date'])

In [84]:
file2= file2.sort_values(by=['Date'])

In [85]:
cols = ["home_team_avg_goals_past", "away_team_avg_goals_past",
        "home_team_avg_goals_last5", "away_team_avg_goals_last5"]
file2[cols] = file2[cols].fillna(file2[cols].mean())

In [ ]:
df1.columns

In [86]:
df1 = file2.drop(columns =['match_id','home_team','away_team','Date'])

In [87]:
df1 = df1.drop(columns =['home_score','away_score','result'])

In [88]:
df1 = df1.replace('', 0).fillna(0)

In [89]:
df1['predicted_result'] = rf1.predict(df1)

In [90]:
df1.to_excel('output28.xlsx', index = False)

<ipython-input-90-29dace1caad4>:1: UserWarning: Pandas requires version '1.4.3' or newer of 'xlsxwriter' (version '1.3.8' currently installed).
  df1.to_excel('output28.xlsx', index = False)
